# Plot linkage and TF-IDF recommendations

This notebook links the CMU Movie Summary Corpus to MovieLens, audits match quality, and evaluates two plot-text recommenders. Run notebook 01 first: it owns the chronological 60/20/20 split **within each user** and writes the canonical interaction files. This notebook reads those files; it never recreates boundaries or uses test feedback.

All variants recommend from movies observed in training and exclude every movie the target user rated in training, regardless of rating. Ratings of at least 4.0 are positive feedback. Validation Recall@10 uses every held-out positive in its denominator, including titles absent from training or without a matched plot. Users without usable plot likes remain in the evaluation and receive the popularity fallback. Mean recall gives each eligible user equal weight.

The experiment is retrospective. Per-user chronology does not establish a shared calendar cutoff across users, and neither the CMU snapshot nor the pretrained features carry historical availability guarantees. Plot matching uses catalog metadata rather than held-out rating labels.

Source: [CMU Movie Summary Corpus](https://www.cs.cmu.edu/~ark/personas/), collected by David Bamman, Brendan O’Connor, and Noah A. Smith. Source license: [CC BY-SA 3.0 US](https://creativecommons.org/licenses/by-sa/3.0/us/). Retain the attribution and license when redistributing plot text.

Required local inputs are `data/raw/ml-latest-small/{movies,ratings}.csv` and `data/raw/MovieSummaries/{movie.metadata.tsv,plot_summaries.txt}`. The Python environment needs pandas, NumPy, SciPy, scikit-learn, and the dependencies used by notebook 01. Outputs are written below the split-specific processed directory to avoid mixing results from different experiments.


## Load the canonical experiment

The shared loader validates raw-file and split-file hashes before reading training and validation. Its provenance fields are attached to saved evaluations. The CMU files are independent content inputs and are loaded separately below.


In [ ]:
from pathlib import Path
import hashlib
import json

import pandas as pd

SPLIT_ID = "per_user_chronological_60_20_20_v1"
POSITIVE_THRESHOLD = 4.0
TOP_K = 10

# Resolve the project from either its root or any nested notebook directory.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "raw" / "ml-latest-small").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run from the project containing data/raw/ml-latest-small.")

raw_dir = data_dir = project_root / "data" / "raw" / "ml-latest-small"
processed_dir = project_root / "data" / "processed" / SPLIT_ID
manifest_path = processed_dir / "split_manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError("Run notebook 01 first to write the canonical split.")
split_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert split_manifest["split_id"] == SPLIT_ID
assert split_manifest["positive_threshold"] == POSITIVE_THRESHOLD
assert split_manifest["top_k"] == TOP_K

def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for filename, expected_hash in split_manifest["raw_sha256"].items():
    if file_sha256(raw_dir / filename) != expected_hash:
        raise ValueError(f"{filename} changed; regenerate the split in notebook 01.")

movies = pd.read_csv(raw_dir / "movies.csv")
loaded_splits = {}
for name in ("train", "validation"):
    path = processed_dir / f"{name}_ratings.csv"
    expected = split_manifest["files"][path.name]
    if file_sha256(path) != expected["sha256"]:
        raise ValueError(f"{path.name} does not match the split manifest.")
    loaded_splits[name] = pd.read_csv(path, parse_dates=["rated_at"])
    assert len(loaded_splits[name]) == expected["rows"]

train_ratings = loaded_splits["train"]
validation_ratings = loaded_splits["validation"]

def save_validation_results(results, filename):
    """Attach the split and evaluation contract to each per-user result row."""
    saved = results.copy()
    saved["split_id"] = SPLIT_ID
    saved["split_fingerprint"] = split_manifest["split_fingerprint"]
    saved["evaluation_split"] = "validation"
    saved["positive_threshold"] = POSITIVE_THRESHOLD
    saved["k"] = TOP_K
    saved.to_csv(processed_dir / filename, index=False)

def validate_result_provenance(results, label="results"):
    expected = {
        "split_id": SPLIT_ID,
        "split_fingerprint": split_manifest["split_fingerprint"],
        "evaluation_split": "validation",
        "positive_threshold": POSITIVE_THRESHOLD,
        "k": TOP_K,
    }
    for column, value in expected.items():
        assert column in results and results[column].eq(value).all(), (
            f"{label}: missing or incompatible {column}; rerun its source notebook."
        )

print("Split:", SPLIT_ID)
print("Training ratings:", len(train_ratings))
print("Validation ratings:", len(validation_ratings))

import csv

cmu_dir = project_root / "data" / "raw" / "MovieSummaries"


## Read CMU metadata

Column names follow the corpus schema. Tab separators and disabled CSV quote handling preserve the source text without interpreting embedded quotation marks as CSV syntax. Only title and release date are needed for linkage.


In [ ]:
metadata_columns = [
    "wikipedia_movie_id",
    "freebase_movie_id",
    "cmu_title",
    "release_date",
    "box_office_revenue",
    "runtime",
    "languages",
    "countries",
    "cmu_genres"
]

cmu_metadata = pd.read_csv(
    cmu_dir / "movie.metadata.tsv",
    sep="\t",
    header=None,
    names=metadata_columns,
    quoting=csv.QUOTE_NONE,
    encoding="utf-8"
)

print("Metadata shape:", cmu_metadata.shape)

cmu_metadata[
    ["wikipedia_movie_id", "cmu_title", "release_date"]
].head()


## Read plot summaries

Each source row contains a Wikipedia movie identifier and narrative text. Inspecting a short excerpt helps catch parsing problems without flooding the notebook with entire plots.


In [ ]:
cmu_plots = pd.read_csv(
    cmu_dir / "plot_summaries.txt",
    sep="\t",
    header=None,
    names=["wikipedia_movie_id", "plot"],
    quoting=csv.QUOTE_NONE,
    encoding="utf-8"
)

print("Plot summaries shape:", cmu_plots.shape)

print("\nFirst plot — first 500 characters:")
print(cmu_plots["plot"].iloc[0][:500])


## Audit source keys and text

Repeated or missing identifiers would invalidate a one-to-one join. Blank plots cannot produce useful text features. Summaries without metadata are counted explicitly because they cannot participate in title/year matching.


In [ ]:
for table_name, table in [
    ("Metadata", cmu_metadata),
    ("Plot summaries", cmu_plots)
]:
    print(table_name)
    print(
        "Missing movie IDs:",
        table["wikipedia_movie_id"].isna().sum()
    )
    print(
        "Extra rows with repeated movie IDs:",
        table["wikipedia_movie_id"].duplicated().sum()
    )
    print()

empty_plot_text = (
    cmu_plots["plot"].fillna("").str.strip().eq("")
)

plots_without_metadata = (
    ~cmu_plots["wikipedia_movie_id"].isin(
        cmu_metadata["wikipedia_movie_id"]
    )
)

print("Missing or blank plot text:", empty_plot_text.sum())
print(
    "Plot rows without matching metadata:",
    plots_without_metadata.sum()
)


## Join summaries to metadata

A left join preserves every plot row, including those lacking metadata. `validate="one_to_one"` fails if source keys violate the expected cardinality; the join indicator preserves the distinction between matched and unmatched records.


In [ ]:
cmu_plot_catalog = cmu_plots.merge(
    cmu_metadata[
        ["wikipedia_movie_id", "cmu_title", "release_date"]
    ],
    on="wikipedia_movie_id",
    how="left",
    validate="one_to_one",
    indicator="metadata_match"
)


## Check join preservation

The row counts and join indicator expose accidental row multiplication or dropped summaries. These figures are computed from the current inputs rather than copied from an earlier run.


In [ ]:
print("Plot rows before join:", len(cmu_plots))
print("Plot rows after join:", len(cmu_plot_catalog))

print(
    "Plots with matching metadata:",
    cmu_plot_catalog["metadata_match"].eq("both").sum()
)

print(
    "Plots without matching metadata:",
    cmu_plot_catalog["metadata_match"].eq("left_only").sum()
)

cmu_plot_catalog[
    ["wikipedia_movie_id", "cmu_title", "release_date", "metadata_match"]
].head()


## Inspect summaries without metadata

These rows remain visible for audit, but lack the title and year needed for the automatic matching rules. They are not guessed into the MovieLens catalog.


In [ ]:
unmatched_plots = cmu_plot_catalog.loc[
    cmu_plot_catalog["metadata_match"] == "left_only",
    ["wikipedia_movie_id", "plot"]
].copy()

unmatched_plots.head()


## Extract release years

Only summaries with matching metadata enter the matching workspace. A four-digit release year is extracted from the source date and stored as a nullable integer, so missing dates remain distinguishable from valid years.


In [ ]:
cmu_match_candidates = cmu_plot_catalog.loc[
    cmu_plot_catalog["metadata_match"] == "both"
].copy()

cmu_match_candidates["release_year"] = (
    cmu_match_candidates["release_date"]
    .astype("string")
    .str.strip()
    .str.extract(r"^(\d{4})(?:-|$)", expand=False)
    .astype("Int64")
)


## Measure matching readiness

The first matching pass requires both a nonblank title and a recognizable year. The audit reports how many source rows are eligible before any title normalization or exclusion.


In [ ]:
has_title = (
    cmu_match_candidates["cmu_title"]
    .fillna("")
    .str.strip()
    .ne("")
)

has_release_year = (
    cmu_match_candidates["release_year"].notna()
)

print("Metadata-matched plots:", len(cmu_match_candidates))
print("Missing or blank titles:", (~has_title).sum())
print("Missing or unrecognized years:", (~has_release_year).sum())
print(
    "Rows with both title and year:",
    (has_title & has_release_year).sum()
)

cmu_match_candidates[
    ["wikipedia_movie_id", "cmu_title", "release_date", "release_year"]
].head()


## Prepare MovieLens titles and years

The terminal parenthesized year in MovieLens is separated from its title. This creates comparable attributes for matching and retains the original title for inspection and display.


In [ ]:
cmu_match_ready = cmu_match_candidates.loc[
    has_title & has_release_year
].copy()

movielens_movies = movies.copy()

movielens_movies["release_year"] = (
    movielens_movies["title"]
    .str.extract(r"\((\d{4})\)\s*$", expand=False)
    .astype("Int64")
)

movielens_movies["title_without_year"] = (
    movielens_movies["title"]
    .str.replace(r"\s*\(\d{4}\)\s*$", "", regex=True)
    .str.strip()
)


## Normalize titles conservatively

Case folding, whitespace normalization, and moving a trailing English article make formatting variants comparable. Punctuation and other content remain intact so the first pass does not collapse unnecessarily different titles.


In [ ]:
import re

def normalize_title(title):
    """Standardize a movie title for matching across datasets."""
    title = title.casefold().strip()
    title = re.sub(r"\s+", " ", title)
    title = re.sub(r"^(.+),\s*(the|a|an)$", r"\2 \1", title)
    return title


cmu_match_ready["title_key"] = (
    cmu_match_ready["cmu_title"].map(normalize_title)
)

movielens_movies["title_key"] = (
    movielens_movies["title_without_year"].map(normalize_title)
)


## Inspect normalized keys

The sample and counts provide an inexpensive check that year extraction and article handling behave as intended in both catalogs. Movies without an extracted year remain in the full catalog but cannot be matched by these rules.


In [ ]:
print("CMU movies prepared:", len(cmu_match_ready))
print("MovieLens movies:", len(movielens_movies))
print(
    "MovieLens movies without an extracted year:",
    movielens_movies["release_year"].isna().sum()
)

display(
    cmu_match_ready[
        ["cmu_title", "title_key", "release_year"]
    ].head()
)

display(
    movielens_movies[
        ["movieId", "title", "title_key", "release_year"]
    ].head()
)


## Select MovieLens rows eligible for matching

Only rows with an extracted release year enter title/year matching. This filtering affects linkage, not the candidate catalog: unmatched movies are restored when the complete plot catalog is assembled.


In [ ]:
movielens_match_ready = movielens_movies.loc[
    movielens_movies["release_year"].notna()
].copy()

print("MovieLens rows ready for matching:", len(movielens_match_ready))


## Detect ambiguous title/year keys

Unique movie IDs do not guarantee unique title/year combinations. Both catalogs are checked for repeated keys before joining; otherwise one title could attach several competing plots.


In [ ]:
match_columns = ["title_key", "release_year"]

cmu_repeated_keys = cmu_match_ready.duplicated(
    subset=match_columns,
    keep=False
)

movielens_repeated_keys = movielens_match_ready.duplicated(
    subset=match_columns,
    keep=False
)

for name, table, repeated_keys in [
    ("CMU", cmu_match_ready, cmu_repeated_keys),
    ("MovieLens", movielens_match_ready, movielens_repeated_keys)
]:
    print(name)
    print(
        "Blank title keys:",
        table["title_key"].fillna("").str.strip().eq("").sum()
    )
    print(
        "Rows belonging to repeated title/year combinations:",
        repeated_keys.sum()
    )
    print()


## Inspect ambiguous keys

The examples make the exclusion rule reviewable. Ambiguous records are excluded from automatic matches instead of choosing whichever source row happens to appear first.


In [ ]:
display(
    cmu_match_ready.loc[
        cmu_repeated_keys,
        ["wikipedia_movie_id", "cmu_title", "title_key", "release_year"]
    ].sort_values(match_columns).head(10)
)

display(
    movielens_match_ready.loc[
        movielens_repeated_keys,
        ["movieId", "title", "title_key", "release_year"]
    ].sort_values(match_columns).head(10)
)


## Keep unambiguous exact-match keys

All occurrences of a repeated key are removed from this matching pass. This sacrifices some coverage to avoid arbitrary many-to-many links; the printed counts document the tradeoff.


In [ ]:
cmu_unique_keys = cmu_match_ready.loc[
    ~cmu_repeated_keys
].copy()

movielens_unique_keys = movielens_match_ready.loc[
    ~movielens_repeated_keys
].copy()

print("CMU rows excluded as ambiguous:", cmu_repeated_keys.sum())
print("MovieLens rows excluded as ambiguous:", movielens_repeated_keys.sum())

print("CMU rows with unique keys:", len(cmu_unique_keys))
print("MovieLens rows with unique keys:", len(movielens_unique_keys))


## Run the exact title/year match

The left join retains eligible MovieLens movies even when a plot is unavailable. One-to-one validation enforces the key policy established above, and the indicator distinguishes missing matches from successful links.


In [ ]:
initial_plot_matches = movielens_unique_keys.merge(
    cmu_unique_keys[
        [
            "title_key",
            "release_year",
            "wikipedia_movie_id",
            "cmu_title",
            "plot"
        ]
    ],
    on=match_columns,
    how="left",
    validate="one_to_one",
    indicator="plot_match"
)


## Measure exact-match coverage

Coverage is reported relative to the entire MovieLens catalog, not just match-ready movies. A few matched titles are shown so formatting errors can be spotted before the second pass.


In [ ]:
matched = initial_plot_matches["plot_match"].eq("both")

print("MovieLens rows before join:", len(movielens_unique_keys))
print("Rows after join:", len(initial_plot_matches))
print("Movies matched to plots:", matched.sum())
print("Eligible movies without a match:", (~matched).sum())

print(
    "Initial coverage of all MovieLens movies:",
    f"{matched.sum() / len(movielens_movies):.1%}"
)

display(
    initial_plot_matches.loc[
        matched,
        ["movieId", "title", "cmu_title", "release_year"]
    ].head(10)
)


## Collect unmatched titles

The unmatched workspace feeds a second, explicitly more permissive matching pass. Original and normalized titles are kept together to make the proposed transformations inspectable.


In [ ]:
unmatched_movielens = initial_plot_matches.loc[
    initial_plot_matches["plot_match"].eq("left_only"),
    ["movieId", "title", "title_without_year", "title_key", "release_year"]
].copy()

display(unmatched_movielens.head(20))


## Review a reproducible unmatched sample

A fixed sampling seed makes the diagnostic stable across reruns. Sampling is capped at the number of available rows so a small input catalog remains valid.


In [ ]:
display(
    unmatched_movielens.sample(
        n=min(15, len(unmatched_movielens)),
        random_state=42
    ).sort_values("release_year")
)


## Create main-title matching keys

Removing parenthetical text can recover alternate-title formatting differences. This transformation is more aggressive than the exact pass, so it is kept separate and subjected to its own ambiguity checks.


In [ ]:
def make_main_title_key(title):
    """Remove parenthetical text, then normalize the remaining title."""
    main_title = re.sub(r"\([^()]*\)", "", title)
    return normalize_title(main_title)


movielens_match_ready["main_title_key"] = (
    movielens_match_ready["title_without_year"].map(make_main_title_key)
)

cmu_match_ready["main_title_key"] = (
    cmu_match_ready["cmu_title"].map(make_main_title_key)
)


## Inspect titles changed by the second rule

Only previously unmatched MovieLens rows are shown. This diagnostic explains what the parenthetical-removal pass can change without using validation performance to alter individual links.


In [ ]:
changed_unmatched = movielens_match_ready.loc[
    movielens_match_ready["movieId"].isin(
        unmatched_movielens["movieId"]
    )
    & movielens_match_ready["main_title_key"].ne(
        movielens_match_ready["title_key"]
    ),
    ["movieId", "title", "title_key", "main_title_key", "release_year"]
]

print("Unmatched movies with a changed key:", len(changed_unmatched))

display(changed_unmatched.head(15))


## Audit the permissive keys

Main-title/year combinations are checked independently in each catalog. Empty and duplicated keys are excluded from the second pass because title simplification can introduce collisions.


In [ ]:
main_match_columns = ["main_title_key", "release_year"]

cmu_main_repeated = cmu_match_ready.duplicated(
    subset=main_match_columns,
    keep=False
)

movielens_main_repeated = movielens_match_ready.duplicated(
    subset=main_match_columns,
    keep=False
)

for name, table, repeated in [
    ("CMU", cmu_match_ready, cmu_main_repeated),
    ("MovieLens", movielens_match_ready, movielens_main_repeated)
]:
    print(name)
    print(
        "Blank main-title keys:",
        table["main_title_key"].eq("").sum()
    )
    print(
        "Rows in repeated main-title/year combinations:",
        repeated.sum()
    )
    print()


## Reserve exact matches before the second pass

Exact matches are retained as the preferred links. The second-pass workspaces contain only eligible unmatched MovieLens movies and CMU plots not already assigned by the exact rule.


In [ ]:
exact_plot_matches = initial_plot_matches.loc[
    initial_plot_matches["plot_match"].eq("both")
].copy()

movielens_second_pass = movielens_match_ready.loc[
    ~movielens_main_repeated
    & movielens_match_ready["main_title_key"].ne("")
    & movielens_match_ready["movieId"].isin(
        unmatched_movielens["movieId"]
    )
].copy()

cmu_second_pass = cmu_match_ready.loc[
    ~cmu_main_repeated
    & cmu_match_ready["main_title_key"].ne("")
    & ~cmu_match_ready["wikipedia_movie_id"].isin(
        exact_plot_matches["wikipedia_movie_id"]
    )
].copy()


## Propose additional links

An inner join produces only main-title/year proposals with an actual source plot. The match-method field preserves how each link was obtained for later auditing.


In [ ]:
main_title_proposals = movielens_second_pass.merge(
    cmu_second_pass[
        [
            "main_title_key",
            "release_year",
            "wikipedia_movie_id",
            "cmu_title",
            "plot"
        ]
    ],
    on=main_match_columns,
    how="inner",
    validate="one_to_one"
)

main_title_proposals["match_method"] = "main_title_year"


## Review additional matches and coverage

The counts quantify the coverage gained by the permissive rule. Paired MovieLens and CMU titles expose potentially incorrect links; title/year agreement reduces ambiguity but is not a guarantee of identity.


In [ ]:
print("Original matches:", len(exact_plot_matches))
print("Additional proposed matches:", len(main_title_proposals))

potential_total = len(exact_plot_matches) + len(main_title_proposals)

print("Potential total matches:", potential_total)
print(
    "Potential coverage of all MovieLens movies:",
    f"{potential_total / len(movielens_movies):.1%}"
)

with pd.option_context("display.max_colwidth", None):
    display(
        main_title_proposals[
            [
                "movieId",
                "title",
                "cmu_title",
                "main_title_key",
                "release_year"
            ]
        ].head(20)
    )


## Combine matching passes

The two disjoint passes are concatenated into one linkage table. A nullable integer preserves the CMU identifier without changing it into floating-point display text.


In [ ]:
exact_plot_matches["match_method"] = "normalized_title_year"

match_output_columns = [
    "movieId",
    "title",
    "release_year",
    "wikipedia_movie_id",
    "cmu_title",
    "plot",
    "match_method"
]

movie_plot_matches = pd.concat(
    [
        exact_plot_matches[match_output_columns],
        main_title_proposals[match_output_columns]
    ],
    ignore_index=True
)

movie_plot_matches["wikipedia_movie_id"] = (
    movie_plot_matches["wikipedia_movie_id"].astype("Int64")
)


## Enforce linkage integrity

Every accepted link must have one MovieLens ID, one distinct CMU ID, and nonblank text. These assertions prevent downstream feature matrices from silently duplicating movies or representing unusable plots.


In [ ]:
assert len(movie_plot_matches) == (
    len(exact_plot_matches) + len(main_title_proposals)
)

assert movie_plot_matches["movieId"].notna().all()
assert movie_plot_matches["wikipedia_movie_id"].notna().all()

assert movie_plot_matches["movieId"].is_unique
assert movie_plot_matches["wikipedia_movie_id"].is_unique

assert (
    movie_plot_matches["plot"]
    .fillna("")
    .str.strip()
    .ne("")
    .all()
)

print("Matched movies:", len(movie_plot_matches))
print(
    "Coverage:",
    f"{len(movie_plot_matches) / len(movielens_movies):.1%}"
)

print("\nMatches by method:")
print(movie_plot_matches["match_method"].value_counts())


## Inspect plots from permissive matches

A reproducible sample includes the paired titles and a narrative excerpt. This is a diagnostic sample rather than a manual correction list; automatic matching can still contain errors outside the sample.


In [ ]:
plot_sample = movie_plot_matches.loc[
    movie_plot_matches["match_method"].eq("main_title_year")
].sample(
    n=min(3, movie_plot_matches["match_method"].eq("main_title_year").sum()),
    random_state=42
)

for row in plot_sample.itertuples(index=False):
    print(f"MovieLens: {row.title}")
    print(f"CMU: {row.cmu_title}")
    print(f"Plot excerpt: {row.plot[:600]}")
    print()


## Report the fitting catalog size

Training interactions come from the canonical per-user split. Candidate membership depends on observed fitting interactions, while plot availability is measured separately.


In [ ]:
print("Training ratings:", len(train_ratings))
print("Training movies:", train_ratings["movieId"].nunique())


## Build full and training plot catalogs

The full catalog retains every MovieLens movie, including unmatched titles. The training catalog is a strict subset selected only by training movie IDs. Missing plots remain explicit rows so every model can recommend from the same candidate universe. The full catalog also supports a later training-plus-validation refit without re-running linkage rules.


In [ ]:
plot_columns = [
    "movieId", "wikipedia_movie_id", "cmu_title", "plot", "match_method"
]
movie_plot_catalog = (
    movies[["movieId", "title", "genres"]]
    .merge(movie_plot_matches[plot_columns], on="movieId", how="left",
           validate="one_to_one")
    .sort_values("movieId")
    .reset_index(drop=True)
)
movie_plot_catalog["has_plot"] = (
    movie_plot_catalog["plot"].fillna("").str.strip().ne("")
)
assert movie_plot_catalog["movieId"].is_unique
assert set(movie_plot_catalog["movieId"]) == set(movies["movieId"])

training_plot_catalog = movie_plot_catalog.loc[
    movie_plot_catalog["movieId"].isin(train_ratings["movieId"])
].copy().reset_index(drop=True)
assert set(training_plot_catalog["movieId"]) == set(train_ratings["movieId"])


## Measure training plot coverage

Coverage is computed for the actual fitting catalog. Missing-plot titles remain recommendation candidates, but receive zero plot similarity; their presence matters when interpreting text-only performance.


In [ ]:
print("Training candidate movies:", len(training_plot_catalog))
print("With plots:", training_plot_catalog["has_plot"].sum())
print("Without plots:", (~training_plot_catalog["has_plot"]).sum())

print(
    "Training candidate plot coverage:",
    f"{training_plot_catalog['has_plot'].mean():.1%}"
)

display(
    training_plot_catalog.loc[
        ~training_plot_catalog["has_plot"],
        ["movieId", "title"]
    ].head(10)
)


## Choose rows that can produce text features

The text matrix contains only nonblank plots, sorted by movie ID. Resetting the row index creates a stable positional mapping between `text_movies` and each sparse feature matrix.


In [ ]:
text_movies = (
    training_plot_catalog.loc[
        training_plot_catalog["has_plot"]
    ]
    .sort_values("movieId")
    .reset_index(drop=True)
    .copy()
)

print("Movies available for text features:", len(text_movies))
assert not text_movies.empty, "No matched training plots are available."


## Fit the original TF-IDF representation

The vocabulary and inverse document frequencies are learned only from plots belonging to training candidates. English stop words are removed, terms must appear in at least two plots, and each movie vector is L2 normalized. No held-out ratings influence these features.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2,
    norm="l2"
)

plot_tfidf = tfidf_vectorizer.fit_transform(text_movies["plot"])

print("TF-IDF matrix shape:", plot_tfidf.shape)
print(
    "Movies with no retained words:",
    (plot_tfidf.getnnz(axis=1) == 0).sum()
)


## Inspect a seed movie’s word weights

Toy Story is used when available; otherwise the first plotted training movie becomes the seed. High weights reveal which words drive this representation and help identify possible name-overlap artifacts.


In [ ]:
seed_movie_id = 1 if text_movies["movieId"].eq(1).any() else int(text_movies.iloc[0]["movieId"])
toy_story_row = text_movies.index[text_movies["movieId"].eq(seed_movie_id)][0]
print("Seed movie:", text_movies.loc[toy_story_row, "title"])

toy_story_weights = pd.Series(
    plot_tfidf[toy_story_row].toarray().ravel(),
    index=tfidf_vectorizer.get_feature_names_out(),
    name="tfidf_weight"
)

toy_story_weights.nlargest(15)


## Calculate item-level cosine similarity

Cosine compares the seed plot with all plotted training movies. For L2-normalized TF-IDF vectors, the dot product and cosine similarity coincide; the self-similarity is a basic numerical check.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

toy_story_similarities = cosine_similarity(
    plot_tfidf[toy_story_row],
    plot_tfidf
).ravel()

print("Similarity scores calculated:", len(toy_story_similarities))
print(
    "Toy Story similarity to itself:",
    toy_story_similarities[toy_story_row]
)


## Rank plot neighbors

The seed is excluded from its own neighbor list. Ties are resolved by training rating count and then movie ID, matching the deterministic ranking policy used for personalized recommendations.


In [ ]:
similar_movies = text_movies[["movieId", "title"]].copy()

similar_movies["similarity"] = toy_story_similarities

training_rating_counts = train_ratings["movieId"].value_counts()

similar_movies["training_rating_count"] = (
    similar_movies["movieId"].map(training_rating_counts)
)

similar_movies = (
    similar_movies.loc[similar_movies["movieId"].ne(seed_movie_id)]
    .sort_values(
        ["similarity", "training_rating_count", "movieId"],
        ascending=[False, False, True]
    )
    .head(10)
    .reset_index(drop=True)
)

similar_movies


## Inspect word contributions

Elementwise multiplication reveals each shared term’s contribution to cosine similarity. The examples include a sequel and unrelated films when those plots are available. Shared names can support useful links or create accidental similarities; the diagnostic does not establish aggregate recommendation quality.


In [ ]:
feature_names = tfidf_vectorizer.get_feature_names_out()

for comparison_movie_id in [3114, 2041, 318]:
    matching_rows = text_movies.index[text_movies["movieId"].eq(comparison_movie_id)]
    if len(matching_rows) == 0:
        print(f"Skipping movie {comparison_movie_id}: no fitted plot features.")
        continue
    comparison_row = matching_rows[0]

    word_products = plot_tfidf[toy_story_row].multiply(
        plot_tfidf[comparison_row]
    )

    contributions = pd.Series(
        word_products.toarray().ravel(),
        index=feature_names,
        name="similarity_contribution"
    )

    print(text_movies.loc[comparison_row, "title"])
    print(f"Total similarity: {contributions.sum():.6f}")

    display(
        contributions.loc[contributions > 0].nlargest(8)
    )


## Construct an example user’s plot history

A user profile uses only training ratings meeting the positive threshold and only those liked movies with available plots. The counts distinguish missing content features from the absence of any positive training history.


In [ ]:
example_user_id = 1

user_history = train_ratings.loc[
    train_ratings["userId"].eq(example_user_id)
]

liked_movie_ids = user_history.loc[
    user_history["rating"] >= POSITIVE_THRESHOLD,
    "movieId"
]

liked_text_rows = text_movies.index[
    text_movies["movieId"].isin(liked_movie_ids)
].to_numpy()

print("Liked training movies:", len(liked_movie_ids))
print("Liked movies with plot features:", len(liked_text_rows))
print(
    "Liked movies without plot features:",
    len(liked_movie_ids) - len(liked_text_rows)
)


## Score the full training candidate catalog

Each plotted candidate receives its mean cosine similarity to the usable liked plots. All previously rated movies are excluded, including disliked ones. Missing-plot candidates receive zero, and training popularity breaks ties or supplies the entire ranking when the profile is empty.


In [ ]:
text_recommendations = training_plot_catalog.loc[
    ~training_plot_catalog["movieId"].isin(user_history["movieId"]),
    ["movieId", "title", "has_plot"]
].copy()

text_recommendations["score"] = 0.0

if len(liked_text_rows) > 0:
    similarities = cosine_similarity(
        plot_tfidf[liked_text_rows],
        plot_tfidf
    )

    text_scores = pd.Series(
        similarities.mean(axis=0),
        index=text_movies["movieId"]
    )

    text_recommendations["score"] = (
        text_recommendations["movieId"]
        .map(text_scores)
        .fillna(0.0)
    )

training_rating_counts = train_ratings["movieId"].value_counts()

text_recommendations["training_rating_count"] = (
    text_recommendations["movieId"].map(training_rating_counts)
)

text_recommendations = (
    text_recommendations.sort_values(
        ["score", "training_rating_count", "movieId"],
        ascending=[False, False, True]
    )
    .head(10)
    .reset_index(drop=True)
)

text_recommendations


## Package the text recommendation rule

The reusable function applies the same history selection, missing-plot handling, seen-item exclusion, and tie policy to every user. Similarity is a ranking score, not a calibrated star-rating prediction or probability of liking the film.


In [ ]:
def recommend_text_based(
    user_id,
    training_data,
    text_movies,
    plot_tfidf,
    candidate_catalog,
    n=10
):
    """Recommend by mean plot similarity, with popularity as a fallback."""

    user_history = training_data.loc[
        training_data["userId"].eq(user_id)
    ]

    liked_movie_ids = user_history.loc[
        user_history["rating"] >= POSITIVE_THRESHOLD,
        "movieId"
    ]

    liked_text_rows = text_movies.index[
        text_movies["movieId"].isin(liked_movie_ids)
    ].to_numpy()

    recommendations = candidate_catalog.loc[
        ~candidate_catalog["movieId"].isin(user_history["movieId"]),
        ["movieId", "title", "has_plot"]
    ].copy()

    recommendations["score"] = 0.0

    if len(liked_text_rows) > 0:
        similarities = cosine_similarity(
            plot_tfidf[liked_text_rows],
            plot_tfidf
        )

        scores = pd.Series(
            similarities.mean(axis=0),
            index=text_movies["movieId"]
        )

        recommendations["score"] = (
            recommendations["movieId"].map(scores).fillna(0.0)
        )

    rating_counts = training_data["movieId"].value_counts()

    recommendations["training_rating_count"] = (
        recommendations["movieId"].map(rating_counts)
    )

    return recommendations.sort_values(
        ["score", "training_rating_count", "movieId"],
        ascending=[False, False, True]
    ).head(n).reset_index(drop=True)


## Inspect example recommendations

The examples demonstrate behavior for two users under the new split. Their outputs are interpreted from their current training histories; neither user is assumed to be a cold-start case.


In [ ]:
for example_user_id in [1, 2]:
    print("Recommendations for user:", example_user_id)

    display(
        recommend_text_based(
            user_id=example_user_id,
            training_data=train_ratings,
            text_movies=text_movies,
            plot_tfidf=plot_tfidf,
            candidate_catalog=training_plot_catalog,
            n=10
        )
    )


## Evaluate original TF-IDF on validation

Every user with at least one positive validation rating is evaluated. All positive held-out movies remain in the denominator, including unavailable training candidates or movies lacking plots. Recommendation assertions check distinct IDs, candidate membership, and exclusion of all seen fitting movies.


In [ ]:
validation_positives = validation_ratings.loc[
    validation_ratings["rating"] >= POSITIVE_THRESHOLD
].copy()

text_based_rows = []

for user_id, user_positives in validation_positives.groupby("userId"):
    positive_movie_ids = user_positives["movieId"]

    recommendations = recommend_text_based(
        user_id=user_id,
        training_data=train_ratings,
        text_movies=text_movies,
        plot_tfidf=plot_tfidf,
        candidate_catalog=training_plot_catalog,
        n=10
    )

    seen_ids = set(train_ratings.loc[
        train_ratings["userId"].eq(user_id), "movieId"
    ])
    recommended_ids = set(recommendations["movieId"])
    assert recommendations["movieId"].is_unique
    assert recommended_ids.isdisjoint(seen_ids)
    assert recommended_ids.issubset(set(training_plot_catalog["movieId"]))
    assert len(recommendations) <= TOP_K

    hits = recommendations["movieId"].isin(
        positive_movie_ids
    ).sum()

    text_based_rows.append({
        "userId": user_id,
        "positive_movie_count": len(positive_movie_ids),
        "hits_at_10": hits,
        "recall_at_10": hits / len(positive_movie_ids)
    })

text_based_results = pd.DataFrame(text_based_rows)

print("Users evaluated:", len(text_based_results))
print(
    "Mean Recall@10:",
    text_based_results["recall_at_10"].mean()
)


## Summarize validation coverage

The history flag is retained for a consistent result schema across models. Under this per-user split, all validation users should have fitting history; a missing history would indicate a broken split contract rather than a separate model cohort.


In [ ]:
text_based_results["has_training_history"] = (
    text_based_results["userId"].isin(train_ratings["userId"])
)

text_group_summary = (
    text_based_results.groupby("has_training_history")
    .agg(
        users=("userId", "size"),
        mean_recall_at_10=("recall_at_10", "mean")
    )
)

text_group_summary
assert text_based_results["has_training_history"].all()


## Fit the logarithmic term-frequency variant

This changes only `sublinear_tf`: a positive term count becomes `1 + log(count)`. The vocabulary thresholds, fitting catalog, normalizations, user scoring, and evaluation cohort stay fixed so the difference isolates repetition weighting.


In [ ]:
tfidf_log_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2,
    norm="l2",
    sublinear_tf=True
)

plot_tfidf_log = tfidf_log_vectorizer.fit_transform(
    text_movies["plot"]
)

print("Log-TF matrix shape:", plot_tfidf_log.shape)


## Evaluate logarithmic TF-IDF

The same recommendation function and held-out positives are used for both text variants. This preserves paired user-level comparisons and prevents an apparent improvement caused by changing eligible users or denominators.


In [ ]:
log_text_rows = []

for user_id, user_positives in validation_positives.groupby("userId"):
    positive_movie_ids = user_positives["movieId"]

    recommendations = recommend_text_based(
        user_id=user_id,
        training_data=train_ratings,
        text_movies=text_movies,
        plot_tfidf=plot_tfidf_log,
        candidate_catalog=training_plot_catalog,
        n=10
    )

    seen_ids = set(train_ratings.loc[
        train_ratings["userId"].eq(user_id), "movieId"
    ])
    recommended_ids = set(recommendations["movieId"])
    assert recommendations["movieId"].is_unique
    assert recommended_ids.isdisjoint(seen_ids)
    assert recommended_ids.issubset(set(training_plot_catalog["movieId"]))
    assert len(recommendations) <= TOP_K

    hits = recommendations["movieId"].isin(
        positive_movie_ids
    ).sum()

    log_text_rows.append({
        "userId": user_id,
        "positive_movie_count": len(positive_movie_ids),
        "hits_at_10": hits,
        "recall_at_10": hits / len(positive_movie_ids)
    })

log_text_results = pd.DataFrame(log_text_rows)

log_text_results["has_training_history"] = (
    log_text_results["userId"].isin(train_ratings["userId"])
)


## Compare current validation aggregates

These means are recomputed after rebuilding the features on the new split. The text variant with the higher validation mean is a candidate for later selection; this notebook makes no claim about test-set performance.


In [ ]:
print("Users evaluated:", len(log_text_results))

print(
    "Original TF-IDF mean Recall@10:",
    text_based_results["recall_at_10"].mean()
)

print(
    "Log-TF mean Recall@10:",
    log_text_results["recall_at_10"].mean()
)

log_text_group_summary = (
    log_text_results.groupby("has_training_history")
    .agg(
        users=("userId", "size"),
        mean_recall_at_10=("recall_at_10", "mean")
    )
)

log_text_group_summary


## Compare user-level changes

A one-to-one user join keeps the comparison paired. The count of improved, worsened, and unchanged users helps distinguish a broad change from a few large gains in the average.


In [ ]:
paired_text_results = text_based_results.loc[
    text_based_results["has_training_history"],
    ["userId", "hits_at_10", "recall_at_10"]
].merge(
    log_text_results[
        ["userId", "hits_at_10", "recall_at_10"]
    ],
    on="userId",
    how="inner",
    suffixes=("_original", "_log"),
    validate="one_to_one"
)

paired_text_results["recall_change"] = (
    paired_text_results["recall_at_10_log"]
    - paired_text_results["recall_at_10_original"]
)

print("Users compared:", len(paired_text_results))

print(
    "Improved:",
    (
        paired_text_results["hits_at_10_log"]
        > paired_text_results["hits_at_10_original"]
    ).sum()
)

print(
    "Worsened:",
    (
        paired_text_results["hits_at_10_log"]
        < paired_text_results["hits_at_10_original"]
    ).sum()
)

print(
    "Unchanged:",
    (
        paired_text_results["hits_at_10_log"]
        == paired_text_results["hits_at_10_original"]
    ).sum()
)


## Inspect users whose recall changed

Sorting the nonzero differences highlights which validation users account for the aggregate change. Identical recall does not imply identical recommendation lists.


In [ ]:
changed_users = paired_text_results.loc[
    paired_text_results["recall_change"].ne(0)
]

display(
    changed_users.sort_values("recall_change")
)


## Persist split-specific content and results

Both plot catalogs and their content hashes are saved with the split fingerprint and CMU source hashes. Per-user evaluation CSVs receive the common provenance fields, so later comparisons can reject stale or incompatible results. The test partition remains unopened here.


In [ ]:
catalog_files = {}
for filename, catalog in (
    ("movie_plot_catalog.csv", movie_plot_catalog),
    ("training_plot_catalog.csv", training_plot_catalog),
):
    path = processed_dir / filename
    catalog.to_csv(path, index=False)
    catalog_files[filename] = {"sha256": file_sha256(path), "rows": len(catalog)}

plot_catalog_manifest = {
    "split_id": SPLIT_ID,
    "split_fingerprint": split_manifest["split_fingerprint"],
    "matching_policy": "unique normalized title/year, then unique main title/year",
    "files": catalog_files,
    "source_sha256": {
        filename: file_sha256(cmu_dir / filename)
        for filename in ("movie.metadata.tsv", "plot_summaries.txt")
    },
}
(processed_dir / "plot_catalog_manifest.json").write_text(
    json.dumps(plot_catalog_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

training_plot_catalog_sha256 = file_sha256(processed_dir / "training_plot_catalog.csv")
for results in (text_based_results, log_text_results):
    results["plot_catalog_sha256"] = training_plot_catalog_sha256
save_validation_results(text_based_results, "tfidf_validation_results.csv")
save_validation_results(log_text_results, "log_tfidf_validation_results.csv")

text_model_summary = pd.DataFrame([
    {"model": "Original TF-IDF", "users": len(text_based_results),
     "mean_recall_at_10": text_based_results["recall_at_10"].mean()},
    {"model": "Log-TF TF-IDF", "users": len(log_text_results),
     "mean_recall_at_10": log_text_results["recall_at_10"].mean()},
]).sort_values("mean_recall_at_10", ascending=False)
display(text_model_summary)
print("Saved plot catalogs, provenance, and both TF-IDF validation results.")


## Review notes and remaining limitations

Use the computed validation summaries to compare the variants; numerical findings from the former global-time split have been removed. Ambiguous title/year keys are excluded, but automated matches may still link the wrong film. Plot coverage is incomplete, and a zero text score is a missing-evidence policy rather than evidence that the user dislikes a film.

The primary metric includes positives that cannot be recommended from the training catalog. This makes the complete retrieval task visible and keeps denominators identical across models; it also means catalog coverage constrains the achievable recall. The per-user split guarantees fitting history, but not a positive movie or a usable plot profile for every user. Popularity handles those empty profiles without removing them from evaluation.

Original and logarithmic TF-IDF are validation experiments. The final model family and settings must be selected before the reserved test partition is evaluated. Refit any learned content statistics on the final fitting catalog when training and validation are combined.
